# $~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~$**SSVEP-BCI**

# **Offline analysis of an OpenBCI recording**

This Notebook can be used to analyze a recorded **SSVEP BCI** `.txt` file from the Github repository named as **ssvep-basic** where different experiment approaches are shown. Along this Notebook the following representations could be shown: 

1. **Temporal Signal**: Time domain after preprocessing the raw signal.
2. **Welch PSD**: frequency domain where harmonic peaks responses are observed.
3. **Summed PSD Energy**: PSD feature extraction per candidate frequency.
4. **CCA Classification**: first canonical correlation is considered ρ (mean ± SD) upon three harmonics with respect to fundamental frequency.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy.signal import welch, sosfiltfilt, butter, iirnotch, tf2sos
from sklearn.cross_decomposition import CCA
import os, re, warnings
warnings.filterwarnings("ignore")


## ***Initial Configuration***

In [5]:
DATA_FILE = "recording.txt" # PASTE HERE THE NAME OF YOUR RECORDING SESSION FILE

fs               = 250               # Sampling frequency Hz (OpenBCI Cyton)

T_SEGMENT = 40                       # s Time of the registered session
T_DISCARD = 1                        # s Discard of first initial and end seconds

SEGMENTS = {
    "Seg1": {"freq": 60/7, "label": "8.57 Hz"},
    "Seg2": {"freq": 60/6, "label": "10 Hz"},
    "Seg3": {"freq": 60/5, "label": "12 Hz"},
    "Seg4": {"freq": 60/4, "label": "15 Hz"},
}
CANDIDATE_FREQS = [m["freq"] for m in SEGMENTS.values()]
EXPERIMENT_LABEL = "SSVEP - Four simultaneous flickering targets"

EXG_CHANNELS  = ["Fp1", "Fp2", "C3", "C4", "P7", "P8", "O1", "O2"]
USED_CHANNELS = ["P7", "P8", "O1", "O2"]

BP_LO, BP_HI       = 7.0, 70.0       # Bandpass filter (Hz)
NOTCH_FUND         = 50              # PLI supression (50 Hz in Europe)
NOTCH_NH, NOTCH_Q  = 3, 30           # number of notch harmonics and quality factor
USE_CAR            = True            # Common Average Reference

WELCH_SEC = 4                        # (s) such that 1/WELCH_SEC = 0.25 Hz resolution
NPERSEG   = fs * WELCH_SEC
NOVERLAP  = NPERSEG // 2

ENERGY_BAND_WIDTH = 0.5              # (Hz) with respect to each harmonic
ENERGY_HARMONICS  = (1, 2, 3)       # CCA harmonics
CCA_HARMONIC_SETTINGS = {           
    "1 harmonic":  [1],
    "2 harmonics": [1, 2],
    "3 harmonics": [1, 2, 3],
}

SAVE_FIGURES = True                  # Saves the figure as a PNG file

CHANNEL_COLORS = {"P7": "purple", "P8": "deeppink", "O1": "navy", "O2": "seagreen"}
FREQ_COLORS = dict(zip(CANDIDATE_FREQS, ["red", "darkorange", "darkgreen", "darkblue"]))

print(f"Welch PSD windows have: {WELCH_SEC}s x {fs}Hz = {NPERSEG} samples and {fs/NPERSEG:.3f} Hz resolution.\n")

print("Loaded Configuration")

Welch PSD windows have: 4s x 250Hz = 1000 samples and 0.250 Hz resolution.

Loaded Configuration


## ***Signal Processing Functions***

In [ ]:
def load_data(path):
    df = pd.read_csv(path, comment="%")
    df.columns = df.columns.str.strip()
    return df

def detect_channels(df):
    return [c for c in df.columns if "EXG Channel" in c][:8]


def valid_channel(values, threshold=0.30):
    return (np.asarray(values) == 0).mean() < threshold


def clean_segment(seg):
    seg = np.asarray(seg, float).copy()
    seg[seg == 0] = np.nan
    nans = np.isnan(seg)
    if nans.mean() > 0.30:
        return None
    idx = np.arange(len(seg))
    seg[nans] = np.interp(idx[nans], idx[~nans], seg[~nans])
    return seg

def build_bandpass(lo=BP_LO, hi=BP_HI, order=4):
    nyq = fs / 2
    return butter(order, [max(lo/nyq, 1e-4), min(hi/nyq, 0.999)],
                  btype="bandpass", output="sos")


def build_notch(fund=NOTCH_FUND, n_harmonics=NOTCH_NH, Q=NOTCH_Q):
    sos_list = []
    for k in range(1, n_harmonics + 1):
        fk = fund * k
        if fk >= fs / 2:
            break
        b, a = iirnotch(fk, Q=Q, fs=fs)
        sos_list.append(tf2sos(b, a))
    return sos_list


_SOS_BP    = build_bandpass()
_SOS_NOTCH = build_notch()


def apply_filters(sig):
    sig = sosfiltfilt(_SOS_BP, sig)
    sig = sosfiltfilt(_SOS_BP, sig)
    for sos_n in _SOS_NOTCH:
        sig = sosfiltfilt(sos_n, sig)
    return sig

def trim_edges(sig, t_discard=T_DISCARD):
    n = int(t_discard * fs)
    return sig[n:-n] if n > 0 else sig

def apply_CAR(sigs_dict):
    if len(sigs_dict) < 2:
        return {}
    M = np.stack(list(sigs_dict.values()), axis=0)
    ref = M.mean(axis=0)
    return {c: M[i] - ref for i, c in enumerate(sigs_dict.keys())}


print("Signal Processing Functions loaded")

## ***Helper Functions***

In [ ]:
def extract_segments(df, n_segs, t_seg=T_SEGMENT):
    n_per_seg = int(t_seg * fs)
    if len(df) < n_per_seg * n_segs:
        n_segs = len(df) // n_per_seg
    return [df.iloc[i*n_per_seg:(i+1)*n_per_seg].reset_index(drop=True)
            for i in range(n_segs)]


def psd_welch(sig):
    if sig is None or len(sig) < NPERSEG:
        return None, None
    return welch(sig, fs=fs, nperseg=NPERSEG, noverlap=NOVERLAP,
                 window="hann", scaling="density")


def normalized_psd(psd):
    mx = psd.max()
    return psd / mx if mx > 0 else psd


def summed_energy_band(f, psd, f0, width=ENERGY_BAND_WIDTH):
    mask = (f >= f0 - width/2) & (f <= f0 + width/2)
    return psd[mask].sum() if mask.any() else 0.0


def summed_harmonics_energy(f, psd, f0, harmonics=ENERGY_HARMONICS):
    return sum(summed_energy_band(f, psd, f0 * k) for k in harmonics)


def windows_not_overlapped(sig, n=NPERSEG):
    k = len(sig) // n
    return [sig[i*n:(i+1)*n] for i in range(k)]


def cca_references(freq, harmonics, n_samples):
    t = np.arange(n_samples) / fs
    comps = []
    for k in harmonics:
        comps.append(np.sin(2*np.pi*freq*k*t))
        comps.append(np.cos(2*np.pi*freq*k*t))
    return np.array(comps).T


def rho_cca(X, Y):
    cca = CCA(n_components=1)
    cca.fit(X, Y)
    Xc, Yc = cca.transform(X, Y)
    return abs(np.corrcoef(Xc[:, 0], Yc[:, 0])[0, 1])


def classify_cca(X, frequencies, harmonics):
    n = X.shape[0]
    return {f0: rho_cca(X, cca_references(f0, harmonics, n)) for f0 in frequencies}


print("Functions loaded")

## ***Load Cell and Preprocessing***

In [ ]:
print(f"Loading: {DATA_FILE}")
df_full = load_data(DATA_FILE)
print(f"  {len(df_full)} samples ({len(df_full)/fs:.1f} s)")

channels = detect_channels(df_full)
POS = {i: (EXG_CHANNELS[i] if i < len(EXG_CHANNELS) else f"ch{i}") for i in range(len(channels))}
NAME_TO_IDX = {name: i for i, name in POS.items()}
print(f"  {len(channels)} channels detected: {[POS[i] for i in range(len(channels))]}")

seg_dfs  = extract_segments(df_full, n_segs=len(SEGMENTS))
seg_keys = list(SEGMENTS.keys())[:len(seg_dfs)]

sigs_processed = {}
for sk, sdf in zip(seg_keys, seg_dfs):
    filtered = {}
    for name in USED_CHANNELS:
        if name not in NAME_TO_IDX:
            continue
        values = sdf[channels[NAME_TO_IDX[name]]].values
        if not valid_channel(values):
            print(f"  {sk}: channel {name} discarded due to bad conductivity")
            continue
        clean = clean_segment(values)
        if clean is None:
            print(f"  {sk}: channel {name} discarded due to being noisy")
            continue
        filtered[name] = trim_edges(apply_filters(clean))

    car = apply_CAR(filtered) if USE_CAR else {}
    active = car if (USE_CAR and car) else filtered
    sigs_processed[sk] = {"signals": active, "raw_filt": filtered, "car": car}

    n = len(next(iter(active.values()))) if active else 0
    ref = "CAR" if active is car and car else "without CAR"
    print(f"  {sk} ({SEGMENTS[sk]['label']}): {len(active)} channels · {n/fs:.1f}s · {ref}")

# Folder automatically generated to gather the PNG figures
OUT_DIR = "figures_" + re.sub(r"[^\w.-]+", "_", os.path.splitext(os.path.basename(DATA_FILE))[0])
if SAVE_FIGURES:
    os.makedirs(OUT_DIR, exist_ok=True)

def save_figure(fig, name):
    if SAVE_FIGURES:
        path = os.path.join(OUT_DIR, name)
        fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
        print(f"  saved at: {path}")

print("\n Data Ready to Analyze")

# ***Temporal Signal Graph***

In [ ]:
def temporal_graph(seg_key):
    sigs = sigs_processed[seg_key]["signals"]
    if not sigs:
        print(f"No data for {seg_key}"); return
    channels_v = list(sigs.keys())
    n = len(channels_v)

    fig, axes = plt.subplots(n, 1, figsize=(14, 2.3*n), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, channels_v):
        t = np.arange(len(sigs[c])) / fs
        color = CHANNEL_COLORS.get(c, "navy")
        ax.plot(t, sigs[c], lw=0.8, color=color)
        ax.set_title(c, fontweight="bold", color=color)
        ax.set_ylabel("µV", fontweight="bold")
        ax.grid(alpha=0.25)
    axes[-1].set_xlabel("Time (s)", fontweight="bold")
    fig.suptitle(
        f"Graph EEG Temporal Domain Signal — {SEGMENTS[seg_key]['label']}\n"
        f"Bandpass {BP_LO}–{BP_HI} Hz and Comb filter implementations",
        fontweight="bold")
    plt.tight_layout()
    save_figure(fig, f"temporal_{seg_key}.png")
    plt.show()


for sk in seg_keys:
    temporal_graph(sk)

# ***Welch PSD Graph***

In [ ]:
def psd_graph(seg_key):
    sigs = sigs_processed[seg_key]["signals"]
    f0   = SEGMENTS[seg_key]["freq"]
    if not sigs:
        print(f"No data for {seg_key}"); return

    fig, ax = plt.subplots(figsize=(12, 6))
    all_psds = []
    f = None
    for c, sig in sigs.items():
        f, psd = psd_welch(sig)
        if psd is None:
            continue
        psd_n = normalized_psd(psd)
        ax.plot(f, psd_n, color=CHANNEL_COLORS.get(c, "navy"), lw=1.4, label=c, alpha=0.85)
        all_psds.append(psd_n)
    if all_psds:
        ax.plot(f, np.mean(all_psds, axis=0), color="black", lw=2.2, label="Average", zorder=10)

    for fm in CANDIDATE_FREQS:
        color   = FREQ_COLORS.get(fm, "red")
        is_target = abs(fm - f0) < 1e-2
        for k in (1, 2, 3, 4):
            ax.axvline(fm*k, color=color, ls="--" if is_target else ":",
                       lw=1.3 if is_target else 0.8, alpha=0.9 if is_target else 0.4)
            if is_target:
                ax.text(fm*k, 1.02, "f0" if k == 1 else f"{k}f0", color=color,
                        fontweight="bold", ha="center", transform=ax.get_xaxis_transform())

    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    ax.set_ylim(0, 1.08)
    ax.set_xlim(0, min(f0*4 + 10, 80))
    ax.set_xlabel("Frequency (Hz)", fontweight="bold")
    ax.set_ylabel("Normalized PSD", fontweight="bold")
    ax.set_title(f"Welch PSD ({WELCH_SEC}s window, 50% overlap) — {SEGMENTS[seg_key]['label']}",
                 fontweight="bold", pad=22)
    ax.grid(which="both", ls="--", alpha=0.35)
    ax.legend(loc="upper right", ncol=3)
    plt.tight_layout()
    save_figure(fig, f"psd_{seg_key}.png")
    plt.show()


for sk in seg_keys:
    psd_graph(sk)

# ***Summed PSD Energy***

In [ ]:
def energy_graph(seg_key):
    sigs = sigs_processed[seg_key]["signals"]
    f0   = SEGMENTS[seg_key]["freq"]
    channels_v = list(sigs.keys())
    if not channels_v:
        print(f"No data for {seg_key}"); return

    frequencies = [f"{cf:.2f}" for cf in CANDIDATE_FREQS]
    fig, axes = plt.subplots(1, len(channels_v), figsize=(4.5*len(channels_v), 4.5), squeeze=False)
    for ai, c in enumerate(channels_v):
        ax = axes[0, ai]
        f, psd = psd_welch(sigs[c])
        summed_energy = [summed_harmonics_energy(f, psd, cf) for cf in CANDIDATE_FREQS]
        colores  = ["crimson" if abs(cf - f0) < 1e-3 else "steelblue" for cf in CANDIDATE_FREQS]
        ax.bar(frequencies, summed_energy, color=colores, edgecolor="black", alpha=0.85)
        ax.set_title(c, fontweight="bold")
        if ai == 0:
            ax.set_ylabel(f"Summed PSD (three harmonics)", fontweight="bold")
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", ls="--", alpha=0.4)
    fig.suptitle(
        f"PSD energy per candidate frequency — {SEGMENTS[seg_key]['label']}\n"
        f"Red equals to target frequency", fontweight="bold")
    plt.tight_layout()
    save_figure(fig, f"energy_{seg_key}.png")
    plt.show()


for sk in seg_keys:
    energy_graph(sk)

# ***CCA Classification (average ρ ± SD)***

In [ ]:
def graph_cca(seg_key):
    sigs = sigs_processed[seg_key]["signals"]
    f0   = SEGMENTS[seg_key]["freq"]
    channels_v = list(sigs.keys())
    if not channels_v:
        print(f"No data for {seg_key}"); return

    settings = list(CCA_HARMONIC_SETTINGS.keys())
    fig, axes = plt.subplots(1, len(settings), figsize=(4.5*len(settings), 5), squeeze=False)

    windows_per_channel = [windows_not_overlapped(sigs[c]) for c in channels_v]
    n_win = len(windows_per_channel[0]) if windows_per_channel else 0
    y_top = 0.0

    for hi, name in enumerate(settings):
        harmonics = CCA_HARMONIC_SETTINGS[name]
        ax = axes[0, hi]
        rho = {cf: [] for cf in CANDIDATE_FREQS}
        for w in range(n_win):
            X = np.stack([windows_per_channel[ci][w] for ci in range(len(channels_v))], axis=1)
            r = classify_cca(X, CANDIDATE_FREQS, harmonics)
            for cf in CANDIDATE_FREQS:
                rho[cf].append(r[cf])

        for i, cf in enumerate(CANDIDATE_FREQS):
            vals = rho[cf]
            media, sd = np.mean(vals), np.std(vals)
            color = "crimson" if abs(cf - f0) < 1e-2 else "steelblue"
            ax.bar(i, media, yerr=sd, color=color, edgecolor="black", alpha=0.85,
                   capsize=5, width=0.6)
            ax.scatter([i]*len(vals), vals, color="black", s=15, zorder=5)
            y_top = max(y_top, media + sd, max(vals) if vals else 0)

        ax.set_xticks(range(len(CANDIDATE_FREQS)))
        ax.set_xticklabels([f"{cf:.2f}" for cf in CANDIDATE_FREQS], rotation=15)
        ax.set_title(name, fontweight="bold")
        ax.grid(axis="y", ls="--", alpha=0.4)
        if hi == 0:
            ax.set_ylabel("CCA ρ (mean ± SD)", fontweight="bold")

    for ax in axes[0]:                          
        ax.set_ylim(0, max(0.7, y_top * 1.15))

    fig.suptitle(
        f"CCA correlation ρ average ± SD (min & max values) per number of harmonics\n"
        f"{SEGMENTS[seg_key]['label']}\n", fontweight="bold")
    plt.tight_layout()
    save_figure(fig, f"cca_{seg_key}.png")
    plt.show()


for sk in seg_keys:
    graph_cca(sk)